# Notebook 07 — One-Year Failure Target (Version 2)

Construct the leakage-safe target at reference quarter-end $t$: `failed_1yr = 1` only for a genuine FDIC failure during days 1–365 after $t$. Version 2 later uses $B_{t-1}, M_{t-1}$, but predictor timing must not shift the label, outcome window, or fixed 2020Q1–2025Q2 test calendar.

Notebook 07 intentionally retains the locked target calendar from 2001Q1. The Version 2 modelling reference restriction to 2001Q2 is applied downstream.

In [ ]:
# Lock Notebook 07 scope, paths and target rules

from pathlib import Path
import importlib.util
import os

import numpy as np
import pandas as pd
from IPython.display import display


# ---------------------------------------------------------
# 1. Mount Google Drive when running in Colab
# ---------------------------------------------------------

try:
    from google.colab import drive

    drive.mount(
        "/content/drive",
        force_remount=False,
    )
except ImportError:
    # Allows controlled local validation outside Colab.
    pass


# ---------------------------------------------------------
# 2. Define project paths
# ---------------------------------------------------------

PROJECT_ROOT = Path(
    os.environ.get(
        "DISSERTATION_PROJECT_ROOT",
        os.environ.get("DATA_FINAL_ROOT", "/content/drive/MyDrive/Data Final"),
    )
).expanduser().resolve()

ANALYSIS_VERSION = "v2_bank_lag1_main"

# Prefer the dedicated V2 configuration while remaining compatible with the
# shared config.py used by the executed Phase 1 notebooks.
explicit_config = os.environ.get("V2_CONFIG_PATH")
config_candidates = [
    Path(explicit_config).expanduser() if explicit_config else None,
    PROJECT_ROOT / "config_v2.py",
    PROJECT_ROOT / "config.py",
    Path.cwd() / "config_v2.py",
    Path.cwd() / "config.py",
]
config_matches = []
for candidate in config_candidates:
    if candidate is not None and candidate.is_file():
        resolved = candidate.resolve()
        if resolved not in config_matches:
            config_matches.append(resolved)
if not config_matches:
    raise FileNotFoundError(
        "No shared configuration was found. Expected config_v2.py or config.py "
        f"under {PROJECT_ROOT}, or set V2_CONFIG_PATH explicitly."
    )
CONFIG_PATH = config_matches[0]

NB04_OUTPUT_DIR = (
    PROJECT_ROOT
    / "Notebook Outputs"
    / "04_Commercial_Bank_Filter"
)

NB06_OUTPUT_DIR = (
    PROJECT_ROOT
    / "Notebook Outputs"
    / ANALYSIS_VERSION
    / "06_FDIC_Failure_Data_Cleaning"
)

NB07_OUTPUT_DIR = (
    PROJECT_ROOT
    / "Notebook Outputs"
    / ANALYSIS_VERSION
    / "07_One_Year_Failure_Target"
)

NB07_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

COMMERCIAL_BANK_PANEL_PATH = (
    NB04_OUTPUT_DIR
    / "ffiec_commercial_bank_panel_2001_2026.csv"
)

VERIFIED_CROSSWALK_PATH = (
    NB04_OUTPUT_DIR
    / "cell_05_fdic_cert_idrssd_crosswalk.csv"
)

GENUINE_FAILURES_PATH = (
    NB06_OUTPUT_DIR
    / "fdic_genuine_bank_failures_2001_2026.parquet"
)

required_paths = [
    CONFIG_PATH,
    COMMERCIAL_BANK_PANEL_PATH,
    VERIFIED_CROSSWALK_PATH,
    GENUINE_FAILURES_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Notebook 07 prerequisites were not found:\n"
        + "\n".join(missing_paths)
    )


# ---------------------------------------------------------
# 3. Read the shared project configuration
# ---------------------------------------------------------

config_specification = importlib.util.spec_from_file_location(
    "dissertation_config",
    CONFIG_PATH,
)

dissertation_config = importlib.util.module_from_spec(
    config_specification
)

config_specification.loader.exec_module(
    dissertation_config
)

TARGET_HORIZON_DAYS = int(
    dissertation_config.TARGET_HORIZON_DAYS
)

ELIGIBLE_BANK_CLASSES = tuple(
    dissertation_config.ELIGIBLE_BANK_CLASSES
)

REFERENCE_CALENDAR_START = pd.Timestamp("2001-06-30")
FINAL_TEST_START = pd.Timestamp("2020-03-31")
FINAL_TEST_END = pd.Timestamp("2025-06-30")
TARGET_BOUNDARY_CHANGE = "NONE"
PREDICTOR_TIMING_POLICY = "B(t-1), M(t-1) -> Y(t, t+365 days]"
TARGET_WINDOW_POLICY = "failed_1yr uses failure during days 1-365 after report_date t"

OUTCOME_OBSERVED_THROUGH = pd.Timestamp(
    getattr(dissertation_config, "OUTCOME_OBSERVED_THROUGH", "2026-07-20")
)


# ---------------------------------------------------------
# 4. Lock the target definition and non-failure policy
# ---------------------------------------------------------

target_definition = pd.DataFrame({
    "rule": [
        "Observation unit",
        "Reference time",
        "Primary failure identifier",
        "Positive target",
        "Negative target",
        "Same-day observations",
        "Post-failure observations",
        "Incomplete outcome windows",
        "V2 predictor timing",
        "Target-timing independence",
        "Notebook 07 calendar",
        "Version 2 reference-calendar start",
        "Final test calendar",
        "Boundary-change policy",
        "Mergers",
        "Voluntary liquidations",
        "Predictor preprocessing",
    ],
    "locked_treatment": [
        "Commercial-bank quarter",
        "Quarter-end report_date t",
        "Verified FDIC CERT, checked against the Notebook 04 CERT-IDRSSD crosswalk",
        "failed_1yr = 1 only when a genuine FDIC failure occurs 1-365 days after report_date t",
        "failed_1yr = 0 when no genuine FDIC failure occurs in the following 365 days",
        "Remove; a same-day event is not a forward-looking target",
        "Remove; observations at or after failure cannot be used for prediction",
        "Derive the last eligible quarter from the recorded outcome cutoff; require 2025Q2",
        PREDICTOR_TIMING_POLICY,
        "Lagging bank predictors must not move report_date t, the label or the outcome window",
        "Retain the locked target handoff from 2001Q1 through 2025Q2",
        "Notebook 08/09 restrict V2 modelling references to 2001Q2; Notebook 07 does not shift labels",
        "2020Q1-2025Q2, exactly 22 reference quarters",
        "No change approved; stop if 2025Q2 lacks a complete 365-day outcome window",
        "Never treat as failures unless the CERT appears in the genuine FDIC failure handoff",
        "Never treat as failures unless the CERT appears in the genuine FDIC failure handoff",
        "None in Notebook 07",
    ],
})

cell_01_checks = {
    "The shared configuration file exists":
        CONFIG_PATH.exists(),

    "The Notebook 04 commercial-bank panel exists":
        COMMERCIAL_BANK_PANEL_PATH.exists(),

    "The verified CERT-IDRSSD crosswalk exists":
        VERIFIED_CROSSWALK_PATH.exists(),

    "The locked Notebook 06 failure handoff exists":
        GENUINE_FAILURES_PATH.exists(),

    "The target horizon is exactly 365 days":
        (TARGET_HORIZON_DAYS == 365),

    "The eligible bank classes are exactly N, NM and SM":
        (ELIGIBLE_BANK_CLASSES == ("N", "NM", "SM")),

    "The outcome-observation cutoff is 2026-07-20":
        (
            OUTCOME_OBSERVED_THROUGH
            == pd.Timestamp("2026-07-20")
        ),

    "The outcome cutoff supports 2025Q2 plus 365 days":
        (
            OUTCOME_OBSERVED_THROUGH
            >= FINAL_TEST_END + pd.Timedelta(days=TARGET_HORIZON_DAYS)
        ),

    "The fixed final test contains exactly 22 reference quarters":
        (
            len(pd.period_range("2020Q1", "2025Q2", freq="Q")) == 22
        ),

    "No target-boundary change has been approved":
        (TARGET_BOUNDARY_CHANGE == "NONE"),

    "Predictor lagging is explicitly independent of target timing":
        target_definition.loc[
            target_definition["rule"].eq("Target-timing independence"),
            "locked_treatment",
        ].str.contains("must not move", regex=False).all(),

    "Mergers and voluntary liquidations are explicitly non-failures":
        (
            target_definition.loc[
                target_definition["rule"].isin(
                    ["Mergers", "Voluntary liquidations"]
                ),
                "locked_treatment",
            ]
            .str.startswith("Never treat as failures")
            .all()
        ),
}

notebook_07_cell_01_gate = pd.DataFrame({
    "check": list(cell_01_checks.keys()),
    "result": [
        "PASS" if passed else "FAIL"
        for passed in cell_01_checks.values()
    ],
    "passed": list(cell_01_checks.values()),
})

target_definition.to_csv(
    NB07_OUTPUT_DIR / "cell_01_target_definition.csv",
    index=False,
)

notebook_07_cell_01_gate.to_csv(
    NB07_OUTPUT_DIR / "cell_01_setup_gate.csv",
    index=False,
)

print("Notebook 07 Cell 1 validation gate:")
display(notebook_07_cell_01_gate)

print("\nLocked target definition and event treatment:")
with pd.option_context("display.max_colwidth", None):
    display(target_definition)

failed_cell_01_checks = [
    check
    for check, passed in cell_01_checks.items()
    if not passed
]

if failed_cell_01_checks:
    raise AssertionError(
        "Notebook 07 Cell 1 blocked. Failed checks: "
        + "; ".join(failed_cell_01_checks)
    )

print("\nCell 1 completed successfully.")
print("Analysis version:", ANALYSIS_VERSION)
print("Predictor/target timing:", PREDICTOR_TIMING_POLICY)
print("Commercial-bank panel:", COMMERCIAL_BANK_PANEL_PATH)
print("Verified crosswalk:", VERIFIED_CROSSWALK_PATH)
print("Genuine failures:", GENUINE_FAILURES_PATH)
print("Output directory:", NB07_OUTPUT_DIR)


In [ ]:
# Load and independently validate all authoritative inputs


# ---------------------------------------------------------
# 1. Confirm Cell 1 passed
# ---------------------------------------------------------

if not notebook_07_cell_01_gate["passed"].all():
    raise AssertionError(
        "Cell 2 blocked because Cell 1 did not pass."
    )


# ---------------------------------------------------------
# 2. Load the three authoritative inputs
# ---------------------------------------------------------

commercial_bank_panel_source = pd.read_csv(
    COMMERCIAL_BANK_PANEL_PATH,
    parse_dates=["report_date"],
    low_memory=False,
)

verified_crosswalk_source = pd.read_csv(
    VERIFIED_CROSSWALK_PATH,
    parse_dates=[
        "first_report_date",
        "last_report_date",
    ],
    low_memory=False,
)

genuine_failures_source = pd.read_parquet(
    GENUINE_FAILURES_PATH
)


# ---------------------------------------------------------
# 3. Validate required schemas and types
# ---------------------------------------------------------

required_panel_columns = {
    "report_date",
    "cert",
    "idrssd",
    "bank_name",
    "fdic_institution_class",
    "commercial_bank_eligible",
}

required_crosswalk_columns = {
    "cert",
    "idrssd",
    "first_report_date",
    "last_report_date",
    "institution_quarter_observations",
    "institution_classes",
}

required_failure_columns = {
    "cert",
    "fail_date",
    "bank_name",
    "resolution",
    "event_classification",
    "outcome_observed_through",
}

missing_panel_columns = sorted(
    required_panel_columns
    - set(commercial_bank_panel_source.columns)
)

missing_crosswalk_columns = sorted(
    required_crosswalk_columns
    - set(verified_crosswalk_source.columns)
)

missing_failure_columns = sorted(
    required_failure_columns
    - set(genuine_failures_source.columns)
)

if missing_panel_columns:
    raise KeyError(
        "Commercial-bank panel is missing columns: "
        + ", ".join(missing_panel_columns)
    )

if missing_crosswalk_columns:
    raise KeyError(
        "Verified crosswalk is missing columns: "
        + ", ".join(missing_crosswalk_columns)
    )

if missing_failure_columns:
    raise KeyError(
        "Genuine-failure handoff is missing columns: "
        + ", ".join(missing_failure_columns)
    )

commercial_bank_panel_source["report_date"] = (
    pd.to_datetime(
        commercial_bank_panel_source["report_date"],
        errors="raise",
    ).dt.normalize()
)

verified_crosswalk_source["first_report_date"] = (
    pd.to_datetime(
        verified_crosswalk_source["first_report_date"],
        errors="raise",
    ).dt.normalize()
)

verified_crosswalk_source["last_report_date"] = (
    pd.to_datetime(
        verified_crosswalk_source["last_report_date"],
        errors="raise",
    ).dt.normalize()
)

genuine_failures_source["fail_date"] = (
    pd.to_datetime(
        genuine_failures_source["fail_date"],
        errors="raise",
    ).dt.normalize()
)

genuine_failures_source["outcome_observed_through"] = (
    pd.to_datetime(
        genuine_failures_source[
            "outcome_observed_through"
        ],
        errors="raise",
    ).dt.normalize()
)

for dataframe in [
    commercial_bank_panel_source,
    verified_crosswalk_source,
    genuine_failures_source,
]:
    dataframe["cert"] = pd.to_numeric(
        dataframe["cert"],
        errors="raise",
    ).astype("int64")

for dataframe in [
    commercial_bank_panel_source,
    verified_crosswalk_source,
]:
    dataframe["idrssd"] = pd.to_numeric(
        dataframe["idrssd"],
        errors="raise",
    ).astype("int64")


# ---------------------------------------------------------
# 4. Create independent input summaries and checks
# ---------------------------------------------------------

panel_first_report_date = (
    commercial_bank_panel_source["report_date"].min()
)

panel_last_report_date = (
    commercial_bank_panel_source["report_date"].max()
)

failure_outcome_dates = (
    genuine_failures_source[
        "outcome_observed_through"
    ].drop_duplicates()
)

input_summary = pd.DataFrame({
    "input": [
        "Notebook 04 commercial-bank panel",
        "Notebook 04 verified CERT-IDRSSD crosswalk",
        "Notebook 06 genuine failures",
    ],
    "observations": [
        len(commercial_bank_panel_source),
        len(verified_crosswalk_source),
        len(genuine_failures_source),
    ],
    "columns": [
        len(commercial_bank_panel_source.columns),
        len(verified_crosswalk_source.columns),
        len(genuine_failures_source.columns),
    ],
    "unique_cert": [
        commercial_bank_panel_source["cert"].nunique(),
        verified_crosswalk_source["cert"].nunique(),
        genuine_failures_source["cert"].nunique(),
    ],
    "first_date": [
        panel_first_report_date,
        verified_crosswalk_source[
            "first_report_date"
        ].min(),
        genuine_failures_source["fail_date"].min(),
    ],
    "last_date": [
        panel_last_report_date,
        verified_crosswalk_source[
            "last_report_date"
        ].max(),
        genuine_failures_source["fail_date"].max(),
    ],
})

cell_02_checks = {
    "The commercial-bank panel contains 600,618 observations":
        (len(commercial_bank_panel_source) == 600618),

    "The commercial-bank panel contains exactly 39 columns":
        (len(commercial_bank_panel_source.columns) == 39),

    "The commercial-bank panel contains 101 report quarters":
        (
            commercial_bank_panel_source[
                "report_date"
            ].nunique()
            == 101
        ),

    "The commercial-bank panel covers 2001-03-31 to 2026-03-31":
        (
            panel_first_report_date
            == pd.Timestamp("2001-03-31")
            and panel_last_report_date
            == pd.Timestamp("2026-03-31")
        ),

    "The panel contains 9,679 unique CERT values":
        (
            commercial_bank_panel_source[
                "cert"
            ].nunique()
            == 9679
        ),

    "The panel contains 9,699 unique RSSDID values":
        (
            commercial_bank_panel_source[
                "idrssd"
            ].nunique()
            == 9699
        ),

    "No CERT-report_date key is duplicated":
        (
            commercial_bank_panel_source.duplicated(
                ["cert", "report_date"]
            ).sum()
            == 0
        ),

    "No RSSDID-report_date key is duplicated":
        (
            commercial_bank_panel_source.duplicated(
                ["idrssd", "report_date"]
            ).sum()
            == 0
        ),

    "Every panel observation belongs to N, NM or SM":
        (
            commercial_bank_panel_source[
                "fdic_institution_class"
            ].isin(ELIGIBLE_BANK_CLASSES).all()
        ),

    "Every panel observation is marked commercial-bank eligible":
        (
            commercial_bank_panel_source[
                "commercial_bank_eligible"
            ].astype("boolean").eq(True).all()
        ),

    "The verified crosswalk has no duplicate CERT-IDRSSD pair":
        (
            verified_crosswalk_source.duplicated(
                ["cert", "idrssd"]
            ).sum()
            == 0
        ),

    "The failure handoff contains exactly 487 records":
        (len(genuine_failures_source) == 487),

    "The failure handoff contains exactly 22 columns":
        (len(genuine_failures_source.columns) == 22),

    "The failure handoff contains 487 unique CERT values":
        (
            genuine_failures_source["cert"].nunique()
            == 487
        ),

    "Every failure record is a genuine FAILURE event":
        (
            genuine_failures_source[
                "resolution"
            ].eq("FAILURE").all()
            and genuine_failures_source[
                "event_classification"
            ].eq("genuine_failure").all()
        ),

    "The failure handoff carries one outcome-observation cutoff":
        (
            len(failure_outcome_dates) == 1
            and failure_outcome_dates.iloc[0]
            == OUTCOME_OBSERVED_THROUGH
        ),
}

notebook_07_cell_02_gate = pd.DataFrame({
    "check": list(cell_02_checks.keys()),
    "result": [
        "PASS" if passed else "FAIL"
        for passed in cell_02_checks.values()
    ],
    "passed": list(cell_02_checks.values()),
})

input_summary.to_csv(
    NB07_OUTPUT_DIR / "cell_02_input_summary.csv",
    index=False,
)

notebook_07_cell_02_gate.to_csv(
    NB07_OUTPUT_DIR / "cell_02_input_validation_gate.csv",
    index=False,
)

print("Notebook 07 Cell 2 validation gate:")
display(notebook_07_cell_02_gate)

print("\nAuthoritative input summary:")
display(input_summary)

failed_cell_02_checks = [
    check
    for check, passed in cell_02_checks.items()
    if not passed
]

if failed_cell_02_checks:
    raise AssertionError(
        "Notebook 07 Cell 2 blocked. Failed checks: "
        + "; ".join(failed_cell_02_checks)
    )

print("\nCell 2 completed successfully.")
print("All three authoritative inputs are validated and unchanged.")


In [ ]:
# Verify the crosswalk and match genuine failures to the panel


# ---------------------------------------------------------
# 1. Confirm Cell 2 passed and preserve source state
# ---------------------------------------------------------

if not notebook_07_cell_02_gate["passed"].all():
    raise AssertionError(
        "Cell 3 blocked because Cell 2 did not pass."
    )

panel_shape_before_cell_03 = (
    commercial_bank_panel_source.shape
)

panel_key_hash_before_cell_03 = int(
    pd.util.hash_pandas_object(
        commercial_bank_panel_source[
            ["report_date", "cert", "idrssd"]
        ],
        index=False,
    ).sum()
)


# ---------------------------------------------------------
# 2. Confirm every retained panel pair is in the crosswalk
# ---------------------------------------------------------

panel_identifier_pairs = (
    commercial_bank_panel_source[
        ["cert", "idrssd"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

crosswalk_identifier_pairs = (
    verified_crosswalk_source[
        ["cert", "idrssd"]
    ]
    .drop_duplicates()
    .assign(crosswalk_pair_verified=True)
)

panel_pair_validation = (
    panel_identifier_pairs.merge(
        crosswalk_identifier_pairs,
        on=["cert", "idrssd"],
        how="left",
        validate="one_to_one",
        sort=False,
    )
)

unverified_panel_pairs = (
    panel_pair_validation.loc[
        panel_pair_validation[
            "crosswalk_pair_verified"
        ].ne(True)
    ]
    .copy()
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 3. Match each genuine failure CERT to panel coverage
# ---------------------------------------------------------

panel_cert_coverage = (
    commercial_bank_panel_source
    .groupby("cert", observed=True, as_index=False)
    .agg(
        matched_panel_quarters=("report_date", "size"),
        first_panel_report_date=("report_date", "min"),
        last_panel_report_date=("report_date", "max"),
        matched_idrssd_count=("idrssd", "nunique"),
    )
)

crosswalk_cert_coverage = (
    verified_crosswalk_source
    .groupby("cert", observed=True, as_index=False)
    .agg(
        verified_crosswalk_pairs=("idrssd", "nunique"),
    )
)

failure_match_audit = (
    genuine_failures_source[
        [
            "cert",
            "fail_date",
            "bank_name",
            "charter_class",
            "state",
        ]
    ]
    .merge(
        crosswalk_cert_coverage,
        on="cert",
        how="left",
        validate="one_to_one",
        sort=False,
    )
    .merge(
        panel_cert_coverage,
        on="cert",
        how="left",
        validate="one_to_one",
        sort=False,
    )
)

failure_match_audit["matched_verified_crosswalk"] = (
    failure_match_audit[
        "verified_crosswalk_pairs"
    ].fillna(0).gt(0)
)

failure_match_audit["matched_commercial_panel"] = (
    failure_match_audit[
        "matched_panel_quarters"
    ].fillna(0).gt(0)
)

failure_match_audit["within_target_calendar"] = (
    failure_match_audit["fail_date"].gt(
        panel_first_report_date
    )
    & failure_match_audit["fail_date"].le(
        OUTCOME_OBSERVED_THROUGH
    )
)

failure_match_audit["initial_match_status"] = np.select(
    [
        failure_match_audit["fail_date"].le(
            panel_first_report_date
        ),
        ~failure_match_audit[
            "matched_verified_crosswalk"
        ],
        ~failure_match_audit[
            "matched_commercial_panel"
        ],
    ],
    [
        "failure_before_or_on_first_panel_report_date",
        "no_verified_crosswalk_record",
        "crosswalk_present_but_not_in_commercial_panel",
    ],
    default="matched_to_verified_commercial_panel",
)

matched_failure_count = int(
    failure_match_audit[
        "matched_commercial_panel"
    ].sum()
)

target_calendar_failure_count = int(
    failure_match_audit[
        "within_target_calendar"
    ].sum()
)

matched_target_calendar_failure_count = int(
    (
        failure_match_audit[
            "within_target_calendar"
        ]
        & failure_match_audit[
            "matched_commercial_panel"
        ]
    ).sum()
)

overall_failed_bank_matching_rate = (
    matched_failure_count
    / len(failure_match_audit)
)

target_calendar_failed_bank_matching_rate = (
    matched_target_calendar_failure_count
    / target_calendar_failure_count
    if target_calendar_failure_count
    else np.nan
)

failure_matching_summary = pd.DataFrame({
    "measure": [
        "Genuine FDIC failed banks",
        "Failed banks with a verified crosswalk record",
        "Failed banks matched to the commercial-bank panel",
        "Overall failed-bank matching rate",
        "Failures within the target calendar",
        "Target-calendar failures matched to the panel",
        "Target-calendar failed-bank matching rate",
        "Unverified retained panel CERT-IDRSSD pairs",
    ],
    "value": [
        len(failure_match_audit),
        int(
            failure_match_audit[
                "matched_verified_crosswalk"
            ].sum()
        ),
        matched_failure_count,
        overall_failed_bank_matching_rate,
        target_calendar_failure_count,
        matched_target_calendar_failure_count,
        target_calendar_failed_bank_matching_rate,
        len(unverified_panel_pairs),
    ],
})


# ---------------------------------------------------------
# 4. Validate and save the matching audit
# ---------------------------------------------------------

cell_03_checks = {
    "Every retained panel CERT-IDRSSD pair is verified by Notebook 04":
        (len(unverified_panel_pairs) == 0),

    "Every matched failure has a verified crosswalk record":
        (
            failure_match_audit.loc[
                failure_match_audit[
                    "matched_commercial_panel"
                ],
                "matched_verified_crosswalk",
            ].all()
        ),

    "Every one of the 487 failures received one match status":
        (
            len(failure_match_audit) == 487
            and failure_match_audit[
                "initial_match_status"
            ].notna().all()
        ),

    "The failure match did not duplicate a CERT":
        failure_match_audit["cert"].is_unique,

    "The matching rates are bounded between zero and one":
        (
            0 <= overall_failed_bank_matching_rate <= 1
            and 0
            <= target_calendar_failed_bank_matching_rate
            <= 1
        ),

    "The commercial-bank source shape is unchanged":
        (
            commercial_bank_panel_source.shape
            == panel_shape_before_cell_03
        ),

    "The commercial-bank source keys are unchanged":
        (
            int(
                pd.util.hash_pandas_object(
                    commercial_bank_panel_source[
                        ["report_date", "cert", "idrssd"]
                    ],
                    index=False,
                ).sum()
            )
            == panel_key_hash_before_cell_03
        ),
}

notebook_07_cell_03_gate = pd.DataFrame({
    "check": list(cell_03_checks.keys()),
    "result": [
        "PASS" if passed else "FAIL"
        for passed in cell_03_checks.values()
    ],
    "passed": list(cell_03_checks.values()),
})

failure_match_audit.to_csv(
    NB07_OUTPUT_DIR / "cell_03_failed_bank_match_audit.csv",
    index=False,
    date_format="%Y-%m-%d",
)

failure_matching_summary.to_csv(
    NB07_OUTPUT_DIR / "cell_03_failure_matching_summary.csv",
    index=False,
)

unverified_panel_pairs.to_csv(
    NB07_OUTPUT_DIR / "cell_03_unverified_panel_pairs.csv",
    index=False,
)

notebook_07_cell_03_gate.to_csv(
    NB07_OUTPUT_DIR / "cell_03_matching_validation_gate.csv",
    index=False,
)

print("Notebook 07 Cell 3 validation gate:")
display(notebook_07_cell_03_gate)

print("\nFailed-bank matching summary:")
display(failure_matching_summary)

print("\nFailure match status counts:")
display(
    failure_match_audit[
        "initial_match_status"
    ].value_counts(dropna=False)
    .rename_axis("match_status")
    .reset_index(name="failed_banks")
)

failed_cell_03_checks = [
    check
    for check, passed in cell_03_checks.items()
    if not passed
]

if failed_cell_03_checks:
    raise AssertionError(
        "Notebook 07 Cell 3 blocked. Failed checks: "
        + "; ".join(failed_cell_03_checks)
    )

print("\nCell 3 completed successfully.")
print("All panel identifiers are backed by the verified crosswalk.")


In [ ]:
# Remove same-day, post-failure and right-censored observations


# ---------------------------------------------------------
# 1. Confirm Cell 3 passed
# ---------------------------------------------------------

if not notebook_07_cell_03_gate["passed"].all():
    raise AssertionError(
        "Cell 4 blocked because Cell 3 did not pass."
    )


# ---------------------------------------------------------
# 2. Derive the latest fully observable report quarter
# ---------------------------------------------------------

latest_eligible_report_date_limit = (
    OUTCOME_OBSERVED_THROUGH
    - pd.Timedelta(days=TARGET_HORIZON_DAYS)
)

latest_eligible_period = (
    latest_eligible_report_date_limit.to_period("Q")
)

if (
    latest_eligible_period.end_time.normalize()
    > latest_eligible_report_date_limit
):
    latest_eligible_period -= 1

LATEST_ELIGIBLE_REPORT_DATE = (
    latest_eligible_period.end_time.normalize()
)

MINIMUM_REQUIRED_OUTCOME_END = (
    FINAL_TEST_END + pd.Timedelta(days=TARGET_HORIZON_DAYS)
)

final_test_reference_periods = pd.period_range(
    "2020Q1",
    "2025Q2",
    freq="Q",
)

v2_calendar_boundary_audit = pd.DataFrame({
    "control": [
        "recorded_outcome_observed_through",
        "minimum_required_outcome_end_for_2025Q2",
        "derived_latest_eligible_report_date",
        "required_final_test_end",
        "final_test_reference_quarters",
        "approved_boundary_change",
        "target_anchor",
        "predictor_timing",
    ],
    "value": [
        OUTCOME_OBSERVED_THROUGH,
        MINIMUM_REQUIRED_OUTCOME_END,
        LATEST_ELIGIBLE_REPORT_DATE,
        FINAL_TEST_END,
        len(final_test_reference_periods),
        TARGET_BOUNDARY_CHANGE,
        "report_date t",
        PREDICTOR_TIMING_POLICY,
    ],
})

if OUTCOME_OBSERVED_THROUGH < MINIMUM_REQUIRED_OUTCOME_END:
    raise AssertionError(
        "The recorded FDIC outcome cutoff cannot support a complete 365-day "
        "window through 2025Q2. Stop; do not shorten the fixed test period silently."
    )


# ---------------------------------------------------------
# 3. Attach the one genuine failure date for each CERT
# ---------------------------------------------------------

failure_date_lookup = (
    genuine_failures_source[
        ["cert", "fail_date", "bank_name"]
    ]
    .rename(
        columns={
            "bank_name": "failed_bank_name",
        }
    )
)

target_working_panel = (
    commercial_bank_panel_source.merge(
        failure_date_lookup,
        on="cert",
        how="left",
        validate="many_to_one",
        sort=False,
    )
)

if len(target_working_panel) != len(
    commercial_bank_panel_source
):
    raise AssertionError(
        "The failure-date merge changed the number of panel rows."
    )

target_working_panel["days_to_failure"] = (
    target_working_panel["fail_date"]
    - target_working_panel["report_date"]
).dt.days.astype("Int64")


# ---------------------------------------------------------
# 4. Create mutually exclusive exclusion decisions
# ---------------------------------------------------------

same_day_failure_mask = (
    target_working_panel["days_to_failure"]
    .eq(0)
    .fillna(False)
    .to_numpy(dtype=bool)
)

post_failure_mask = (
    target_working_panel["days_to_failure"]
    .lt(0)
    .fillna(False)
    .to_numpy(dtype=bool)
)

incomplete_outcome_window_mask = (
    target_working_panel["report_date"].gt(
        LATEST_ELIGIBLE_REPORT_DATE
    )
    .to_numpy(dtype=bool)
)

target_working_panel["target_row_decision"] = np.select(
    [
        same_day_failure_mask,
        post_failure_mask,
        incomplete_outcome_window_mask,
    ],
    [
        "EXCLUDE_SAME_DAY_FAILURE",
        "EXCLUDE_POST_FAILURE",
        "EXCLUDE_INCOMPLETE_365_DAY_WINDOW",
    ],
    default="RETAIN_FOR_TARGET",
)

target_exclusion_audit = (
    target_working_panel.loc[
        target_working_panel[
            "target_row_decision"
        ].ne("RETAIN_FOR_TARGET"),
        [
            "report_date",
            "cert",
            "idrssd",
            "bank_name",
            "fail_date",
            "days_to_failure",
            "target_row_decision",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

target_eligible_panel = (
    target_working_panel.loc[
        target_working_panel[
            "target_row_decision"
        ].eq("RETAIN_FOR_TARGET")
    ]
    .copy()
    .reset_index(drop=True)
)

exclusion_summary = (
    target_working_panel[
        "target_row_decision"
    ]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="bank_quarters")
)

exclusion_summary["percentage_of_input"] = (
    exclusion_summary["bank_quarters"]
    / len(target_working_panel)
)


# ---------------------------------------------------------
# 5. Validate and save the exclusions
# ---------------------------------------------------------

cell_04_checks = {
    "The latest eligible report quarter is 2025Q2":
        (
            LATEST_ELIGIBLE_REPORT_DATE
            == pd.Timestamp("2025-06-30")
        ),

    "The recorded outcome cutoff reaches at least 2026-06-30":
        (OUTCOME_OBSERVED_THROUGH >= pd.Timestamp("2026-06-30")),

    "The outcome cutoff supports a complete window through 2025Q2":
        (OUTCOME_OBSERVED_THROUGH >= MINIMUM_REQUIRED_OUTCOME_END),

    "The fixed final test contains exactly 22 reference quarters":
        (len(final_test_reference_periods) == 22),

    "No boundary change has been approved":
        (TARGET_BOUNDARY_CHANGE == "NONE"),

    "Every input observation received exactly one decision":
        (
            target_working_panel[
                "target_row_decision"
            ].notna().all()
        ),

    "Retained and excluded rows reconcile to 600,618":
        (
            len(target_eligible_panel)
            + len(target_exclusion_audit)
            == len(target_working_panel)
            == 600618
        ),

    "No retained observation is on the failure date":
        (
            ~target_eligible_panel[
                "days_to_failure"
            ].eq(0).any()
        ),

    "No retained observation is after failure":
        (
            ~target_eligible_panel[
                "days_to_failure"
            ].lt(0).any()
        ),

    "No retained report date is later than 2025-06-30":
        (
            target_eligible_panel[
                "report_date"
            ].le(LATEST_ELIGIBLE_REPORT_DATE).all()
        ),

    "Every excluded row has an explicit exclusion reason":
        (
            target_exclusion_audit[
                "target_row_decision"
            ].str.startswith("EXCLUDE_").all()
        ),

    "No bank-quarter key was duplicated by the failure merge":
        (
            target_working_panel.duplicated(
                ["cert", "report_date"]
            ).sum()
            == 0
        ),
}

notebook_07_cell_04_gate = pd.DataFrame({
    "check": list(cell_04_checks.keys()),
    "result": [
        "PASS" if passed else "FAIL"
        for passed in cell_04_checks.values()
    ],
    "passed": list(cell_04_checks.values()),
})

v2_calendar_boundary_audit.to_csv(
    NB07_OUTPUT_DIR / "cell_04_v2_calendar_boundary_audit.csv",
    index=False,
)

target_exclusion_audit.to_csv(
    NB07_OUTPUT_DIR / "cell_04_target_exclusion_audit.csv",
    index=False,
    date_format="%Y-%m-%d",
)

exclusion_summary.to_csv(
    NB07_OUTPUT_DIR / "cell_04_exclusion_summary.csv",
    index=False,
)

notebook_07_cell_04_gate.to_csv(
    NB07_OUTPUT_DIR / "cell_04_exclusion_validation_gate.csv",
    index=False,
)

print("Notebook 07 Cell 4 validation gate:")
display(notebook_07_cell_04_gate)

print("\nMutually exclusive observation decisions:")
display(exclusion_summary)

failed_cell_04_checks = [
    check
    for check, passed in cell_04_checks.items()
    if not passed
]

if failed_cell_04_checks:
    raise AssertionError(
        "Notebook 07 Cell 4 blocked. Failed checks: "
        + "; ".join(failed_cell_04_checks)
    )

print("\nCell 4 completed successfully.")
print("Same-day, post-failure and incomplete-window rows are excluded.")


In [ ]:
# Construct failed_1yr and audit failure capture


# ---------------------------------------------------------
# 1. Confirm Cell 4 passed
# ---------------------------------------------------------

if not notebook_07_cell_04_gate["passed"].all():
    raise AssertionError(
        "Cell 5 blocked because Cell 4 did not pass."
    )


# ---------------------------------------------------------
# 2. Apply the locked 1-365-day target rule
# ---------------------------------------------------------

target_eligible_panel["failed_1yr"] = (
    target_eligible_panel[
        "days_to_failure"
    ].between(
        1,
        TARGET_HORIZON_DAYS,
        inclusive="both",
    )
    .fillna(False)
    .astype("int8")
)

target_eligible_panel["target_reason"] = np.select(
    [
        target_eligible_panel["failed_1yr"].eq(1),
        target_eligible_panel["fail_date"].isna(),
        target_eligible_panel[
            "days_to_failure"
        ]
        .gt(TARGET_HORIZON_DAYS)
        .fillna(False)
        .to_numpy(dtype=bool),
    ],
    [
        "genuine_failure_in_1_to_365_days",
        "no_genuine_fdic_failure_for_cert",
        "genuine_failure_more_than_365_days_ahead",
    ],
    default="ERROR_UNCLASSIFIED_TARGET",
)


# ---------------------------------------------------------
# 3. Build positive-horizon and failed-bank capture audits
# ---------------------------------------------------------

positive_target_rows = (
    target_eligible_panel.loc[
        target_eligible_panel["failed_1yr"].eq(1),
        [
            "report_date",
            "cert",
            "idrssd",
            "bank_name",
            "fail_date",
            "days_to_failure",
        ],
    ]
    .copy()
    .sort_values(
        ["fail_date", "cert", "report_date"]
    )
    .reset_index(drop=True)
)

positive_by_failed_bank = (
    positive_target_rows
    .groupby("cert", observed=True, as_index=False)
    .agg(
        positive_bank_quarters=("report_date", "size"),
        earliest_positive_report_date=("report_date", "min"),
        latest_positive_report_date=("report_date", "max"),
        minimum_positive_horizon_days=("days_to_failure", "min"),
        maximum_positive_horizon_days=("days_to_failure", "max"),
    )
)

failure_capture_audit = (
    failure_match_audit.merge(
        positive_by_failed_bank,
        on="cert",
        how="left",
        validate="one_to_one",
        sort=False,
    )
)

failure_capture_audit["positive_bank_quarters"] = (
    failure_capture_audit[
        "positive_bank_quarters"
    ].fillna(0).astype("int64")
)

failure_capture_audit["captured_by_target"] = (
    failure_capture_audit[
        "positive_bank_quarters"
    ].gt(0)
)

failure_capture_audit["final_capture_status"] = np.select(
    [
        failure_capture_audit["fail_date"].le(
            panel_first_report_date
        ),
        ~failure_capture_audit[
            "matched_commercial_panel"
        ],
        failure_capture_audit[
            "captured_by_target"
        ],
    ],
    [
        "not_capturable_failure_before_or_on_first_panel_report",
        "not_captured_no_commercial_panel_match",
        "captured_by_at_least_one_positive_bank_quarter",
    ],
    default="matched_but_no_report_within_365_days",
)

positive_bank_quarters = len(positive_target_rows)
distinct_failed_banks_captured = (
    positive_target_rows["cert"].nunique()
)

target_prevalence = (
    positive_bank_quarters
    / len(target_eligible_panel)
)

target_calendar_capture_rate = (
    int(
        (
            failure_capture_audit[
                "within_target_calendar"
            ]
            & failure_capture_audit[
                "captured_by_target"
            ]
        ).sum()
    )
    / target_calendar_failure_count
    if target_calendar_failure_count
    else np.nan
)

target_construction_summary = pd.DataFrame({
    "measure": [
        "Eligible bank-quarter observations",
        "Positive bank-quarters",
        "Negative bank-quarters",
        "Distinct failed banks captured",
        "Target prevalence",
        "Overall failed-bank panel matching rate",
        "Target-calendar failed-bank panel matching rate",
        "Target-calendar failed-bank capture rate",
        "Minimum positive horizon days",
        "Maximum positive horizon days",
    ],
    "value": [
        len(target_eligible_panel),
        positive_bank_quarters,
        len(target_eligible_panel) - positive_bank_quarters,
        distinct_failed_banks_captured,
        target_prevalence,
        overall_failed_bank_matching_rate,
        target_calendar_failed_bank_matching_rate,
        target_calendar_capture_rate,
        positive_target_rows[
            "days_to_failure"
        ].min(),
        positive_target_rows[
            "days_to_failure"
        ].max(),
    ],
})


# ---------------------------------------------------------
# 4. Validate and save target audits
# ---------------------------------------------------------

cell_05_checks = {
    "failed_1yr contains only zero and one":
        (
            set(
                target_eligible_panel[
                    "failed_1yr"
                ].unique()
            ).issubset({0, 1})
        ),

    "failed_1yr has no missing value":
        target_eligible_panel["failed_1yr"].notna().all(),

    "days_to_failure is anchored exactly at report_date t":
        (
            target_eligible_panel["days_to_failure"]
            .eq(
                (
                    target_eligible_panel["fail_date"]
                    - target_eligible_panel["report_date"]
                ).dt.days.astype("Int64")
            )
            .fillna(True)
            .all()
        ),

    "No predictor-source date enters target construction":
        ({"source_report_date", "expected_bank_source_date"}.isdisjoint(target_eligible_panel.columns)),

    "Every positive horizon is at least one day":
        positive_target_rows[
            "days_to_failure"
        ].ge(1).all(),

    "No positive horizon exceeds 365 days":
        positive_target_rows[
            "days_to_failure"
        ].le(TARGET_HORIZON_DAYS).all(),

    "Every 1-365-day failure horizon is labelled positive":
        (
            target_eligible_panel.loc[
                target_eligible_panel[
                    "days_to_failure"
                ].between(1, TARGET_HORIZON_DAYS),
                "failed_1yr",
            ].eq(1).all()
        ),

    "Every zero target has a valid negative reason":
        (
            target_eligible_panel.loc[
                target_eligible_panel[
                    "failed_1yr"
                ].eq(0),
                "target_reason",
            ].isin(
                [
                    "no_genuine_fdic_failure_for_cert",
                    "genuine_failure_more_than_365_days_ahead",
                ]
            ).all()
        ),

    "Every positive CERT is a genuine FDIC failure CERT":
        (
            set(positive_target_rows["cert"])
            .issubset(
                set(genuine_failures_source["cert"])
            )
        ),

    "Positive and negative observations reconcile":
        (
            positive_bank_quarters
            + int(
                target_eligible_panel[
                    "failed_1yr"
                ].eq(0).sum()
            )
            == len(target_eligible_panel)
        ),

    "No target reason remains unclassified":
        (
            ~target_eligible_panel[
                "target_reason"
            ].eq("ERROR_UNCLASSIFIED_TARGET").any()
        ),

    "Failure capture status is complete for all 487 failures":
        (
            len(failure_capture_audit) == 487
            and failure_capture_audit[
                "final_capture_status"
            ].notna().all()
        ),
}

notebook_07_cell_05_gate = pd.DataFrame({
    "check": list(cell_05_checks.keys()),
    "result": [
        "PASS" if passed else "FAIL"
        for passed in cell_05_checks.values()
    ],
    "passed": list(cell_05_checks.values()),
})

positive_target_rows.to_csv(
    NB07_OUTPUT_DIR / "cell_05_positive_bank_quarters.csv",
    index=False,
    date_format="%Y-%m-%d",
)

failure_capture_audit.to_csv(
    NB07_OUTPUT_DIR / "cell_05_failed_bank_capture_audit.csv",
    index=False,
    date_format="%Y-%m-%d",
)

target_construction_summary.to_csv(
    NB07_OUTPUT_DIR / "cell_05_target_construction_summary.csv",
    index=False,
)

notebook_07_cell_05_gate.to_csv(
    NB07_OUTPUT_DIR / "cell_05_target_validation_gate.csv",
    index=False,
)

print("Notebook 07 Cell 5 validation gate:")
display(notebook_07_cell_05_gate)

print("\nOne-year target construction summary:")
display(target_construction_summary)

print("\nFailed-bank capture status:")
display(
    failure_capture_audit[
        "final_capture_status"
    ].value_counts(dropna=False)
    .rename_axis("capture_status")
    .reset_index(name="failed_banks")
)

failed_cell_05_checks = [
    check
    for check, passed in cell_05_checks.items()
    if not passed
]

if failed_cell_05_checks:
    raise AssertionError(
        "Notebook 07 Cell 5 blocked. Failed checks: "
        + "; ".join(failed_cell_05_checks)
    )

print("\nCell 5 completed successfully.")
print("failed_1yr is constructed using the exact 1-365-day rule.")


In [ ]:
# Document non-failure exits, mergers and voluntary liquidations


# ---------------------------------------------------------
# 1. Confirm Cell 5 passed
# ---------------------------------------------------------

if not notebook_07_cell_05_gate["passed"].all():
    raise AssertionError(
        "Cell 6 blocked because Cell 5 did not pass."
    )


# ---------------------------------------------------------
# 2. Identify panel exits without a genuine FDIC failure
# ---------------------------------------------------------

last_panel_observation_by_cert = (
    commercial_bank_panel_source
    .sort_values(
        ["cert", "report_date", "idrssd"],
        kind="mergesort",
    )
    .groupby("cert", observed=True, as_index=False)
    .tail(1)
    [
        [
            "cert",
            "idrssd",
            "bank_name",
            "report_date",
            "fdic_institution_class",
        ]
    ]
    .rename(
        columns={
            "report_date": "last_panel_report_date",
        }
    )
    .reset_index(drop=True)
)

genuine_failure_certs = set(
    genuine_failures_source["cert"]
)

nonfailure_panel_exits = (
    last_panel_observation_by_cert.loc[
        last_panel_observation_by_cert[
            "last_panel_report_date"
        ].lt(panel_last_report_date)
        & ~last_panel_observation_by_cert[
            "cert"
        ].isin(genuine_failure_certs)
    ]
    .copy()
    .reset_index(drop=True)
)

nonfailure_panel_exits["event_treatment"] = (
    "NON_FAILURE_PANEL_EXIT"
)

nonfailure_panel_exits["interpretation"] = (
    "Panel exit without a genuine FDIC failure record. "
    "This may reflect a merger, voluntary liquidation, charter change, "
    "or another coverage exit; the available inputs do not identify "
    "which mechanism occurred, so no failure label is assigned."
)


# ---------------------------------------------------------
# 3. Create an explicit event-treatment policy table
# ---------------------------------------------------------

event_treatment_policy = pd.DataFrame({
    "event_or_status": [
        "Genuine FDIC failure 1-365 days after report date",
        "Genuine FDIC failure more than 365 days after report date",
        "Same-day FDIC failure",
        "Post-failure bank-quarter",
        "Merger",
        "Voluntary liquidation",
        "Other panel exit without a genuine FDIC failure record",
    ],
    "failed_1yr_treatment": [
        "1",
        "0",
        "Observation excluded",
        "Observation excluded",
        "0; never inferred as failure from panel disappearance",
        "0; never inferred as failure from panel disappearance",
        "0; never inferred as failure from panel disappearance",
    ],
    "evidence_rule": [
        "CERT appears in the locked Notebook 06 genuine-failure handoff",
        "CERT appears in the locked Notebook 06 genuine-failure handoff",
        "CERT appears in the locked Notebook 06 genuine-failure handoff",
        "CERT appears in the locked Notebook 06 genuine-failure handoff",
        "Not present as a genuine FDIC failure event",
        "Not present as a genuine FDIC failure event",
        "Not present as a genuine FDIC failure event",
    ],
})

nonfailure_exit_cert_set = set(
    nonfailure_panel_exits["cert"]
)

positive_cert_set = set(
    positive_target_rows["cert"]
)

cell_06_checks = {
    "Every non-failure exit occurs before the final source quarter":
        (
            nonfailure_panel_exits[
                "last_panel_report_date"
            ].lt(panel_last_report_date).all()
        ),

    "No non-failure exit CERT is in the genuine-failure handoff":
        nonfailure_exit_cert_set.isdisjoint(
            genuine_failure_certs
        ),

    "No non-failure exit CERT receives a positive target":
        nonfailure_exit_cert_set.isdisjoint(
            positive_cert_set
        ),

    "Mergers are explicitly documented as non-failures":
        (
            event_treatment_policy.loc[
                event_treatment_policy[
                    "event_or_status"
                ].eq("Merger"),
                "failed_1yr_treatment",
            ].str.startswith("0;").all()
        ),

    "Voluntary liquidations are explicitly documented as non-failures":
        (
            event_treatment_policy.loc[
                event_treatment_policy[
                    "event_or_status"
                ].eq("Voluntary liquidation"),
                "failed_1yr_treatment",
            ].str.startswith("0;").all()
        ),

    "No panel exit is misclassified from disappearance alone":
        (
            nonfailure_panel_exits[
                "interpretation"
            ].str.contains(
                "no failure label is assigned",
                regex=False,
            ).all()
        ),
}

notebook_07_cell_06_gate = pd.DataFrame({
    "check": list(cell_06_checks.keys()),
    "result": [
        "PASS" if passed else "FAIL"
        for passed in cell_06_checks.values()
    ],
    "passed": list(cell_06_checks.values()),
})

nonfailure_panel_exits.to_csv(
    NB07_OUTPUT_DIR / "cell_06_nonfailure_panel_exits.csv",
    index=False,
    date_format="%Y-%m-%d",
)

event_treatment_policy.to_csv(
    NB07_OUTPUT_DIR / "cell_06_event_treatment_policy.csv",
    index=False,
)

notebook_07_cell_06_gate.to_csv(
    NB07_OUTPUT_DIR / "cell_06_nonfailure_event_gate.csv",
    index=False,
)

print("Notebook 07 Cell 6 validation gate:")
display(notebook_07_cell_06_gate)

print("\nEvent-treatment policy:")
with pd.option_context("display.max_colwidth", None):
    display(event_treatment_policy)

print(
    "\nPanel exits without a genuine FDIC failure record:",
    f"{len(nonfailure_panel_exits):,}",
)

failed_cell_06_checks = [
    check
    for check, passed in cell_06_checks.items()
    if not passed
]

if failed_cell_06_checks:
    raise AssertionError(
        "Notebook 07 Cell 6 blocked. Failed checks: "
        + "; ".join(failed_cell_06_checks)
    )

print("\nCell 6 completed successfully.")
print("Mergers and voluntary liquidations are documented as non-failures.")


In [ ]:
# Save and validate the definitive Notebook 07 handoffs


# ---------------------------------------------------------
# 1. Confirm Cell 6 passed
# ---------------------------------------------------------

if not notebook_07_cell_06_gate["passed"].all():
    raise AssertionError(
        "Cell 7 blocked because Cell 6 did not pass."
    )


# ---------------------------------------------------------
# 2. Create leakage-safe primary and separate audit handoffs
# ---------------------------------------------------------

primary_target_columns = [
    "report_date",
    "cert",
    "idrssd",
    "failed_1yr",
]

audit_target_columns = [
    "report_date",
    "cert",
    "idrssd",
    "failed_1yr",
    "fail_date",
    "days_to_failure",
    "target_reason",
]

notebook_07_primary_target_handoff = (
    target_eligible_panel[
        primary_target_columns
    ]
    .sort_values(
        ["report_date", "cert", "idrssd"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

notebook_07_target_audit_handoff = (
    target_eligible_panel[
        audit_target_columns
    ]
    .sort_values(
        ["report_date", "cert", "idrssd"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 3. Save Parquet handoffs
# ---------------------------------------------------------

PRIMARY_TARGET_PATH = (
    NB07_OUTPUT_DIR
    / "bank_failure_target_1yr_2001Q1_2025Q2.parquet"
)

TARGET_AUDIT_PATH = (
    NB07_OUTPUT_DIR
    / "bank_failure_target_1yr_audit_2001Q1_2025Q2.parquet"
)

notebook_07_primary_target_handoff.to_parquet(
    PRIMARY_TARGET_PATH,
    index=False,
    compression="snappy",
)

notebook_07_target_audit_handoff.to_parquet(
    TARGET_AUDIT_PATH,
    index=False,
    compression="snappy",
)


# ---------------------------------------------------------
# 4. Re-read and verify both written files
# ---------------------------------------------------------

primary_recheck = pd.read_parquet(
    PRIMARY_TARGET_PATH
)

audit_recheck = pd.read_parquet(
    TARGET_AUDIT_PATH
)

primary_memory_hash = int(
    pd.util.hash_pandas_object(
        notebook_07_primary_target_handoff,
        index=False,
    ).sum()
)

primary_file_hash = int(
    pd.util.hash_pandas_object(
        primary_recheck,
        index=False,
    ).sum()
)

audit_memory_key_hash = int(
    pd.util.hash_pandas_object(
        notebook_07_target_audit_handoff[
            primary_target_columns
        ],
        index=False,
    ).sum()
)

audit_file_key_hash = int(
    pd.util.hash_pandas_object(
        audit_recheck[
            primary_target_columns
        ],
        index=False,
    ).sum()
)


# ---------------------------------------------------------
# 5. Build final summary, manifest and validation gate
# ---------------------------------------------------------

decision_count_lookup = (
    target_working_panel[
        "target_row_decision"
    ].value_counts().to_dict()
)

notebook_07_final_summary = pd.DataFrame({
    "measure": [
        "Commercial-bank input observations",
        "Genuine FDIC failure records",
        "Verified panel CERT-IDRSSD pairs",
        "Same-day observations excluded",
        "Post-failure observations excluded",
        "Incomplete 365-day windows excluded",
        "Total observations excluded",
        "Final target observations",
        "Final target columns",
        "Positive bank-quarters",
        "Negative bank-quarters",
        "Distinct failed banks captured",
        "Target prevalence",
        "Overall failed-bank panel matching rate",
        "Target-calendar failed-bank panel matching rate",
        "Target-calendar failed-bank capture rate",
        "Minimum positive horizon days",
        "Maximum positive horizon days",
        "First eligible report date",
        "Last eligible report date",
        "Outcome observed through",
        "Non-failure panel exits documented",
        "Primary target handoff",
        "Separate leakage-sensitive audit handoff",
    ],
    "value": [
        len(commercial_bank_panel_source),
        len(genuine_failures_source),
        len(panel_identifier_pairs),
        decision_count_lookup.get(
            "EXCLUDE_SAME_DAY_FAILURE", 0
        ),
        decision_count_lookup.get(
            "EXCLUDE_POST_FAILURE", 0
        ),
        decision_count_lookup.get(
            "EXCLUDE_INCOMPLETE_365_DAY_WINDOW", 0
        ),
        len(target_exclusion_audit),
        len(notebook_07_primary_target_handoff),
        len(notebook_07_primary_target_handoff.columns),
        positive_bank_quarters,
        len(notebook_07_primary_target_handoff)
        - positive_bank_quarters,
        distinct_failed_banks_captured,
        target_prevalence,
        overall_failed_bank_matching_rate,
        target_calendar_failed_bank_matching_rate,
        target_calendar_capture_rate,
        positive_target_rows["days_to_failure"].min(),
        positive_target_rows["days_to_failure"].max(),
        notebook_07_primary_target_handoff[
            "report_date"
        ].min(),
        notebook_07_primary_target_handoff[
            "report_date"
        ].max(),
        OUTCOME_OBSERVED_THROUGH,
        len(nonfailure_panel_exits),
        PRIMARY_TARGET_PATH.name,
        TARGET_AUDIT_PATH.name,
    ],
})

notebook_07_output_manifest = pd.DataFrame({
    "output_file": [
        PRIMARY_TARGET_PATH.name,
        TARGET_AUDIT_PATH.name,
        "cell_03_failed_bank_match_audit.csv",
        "cell_04_target_exclusion_audit.csv",
        "cell_05_failed_bank_capture_audit.csv",
        "cell_05_positive_bank_quarters.csv",
        "cell_06_nonfailure_panel_exits.csv",
        "cell_06_event_treatment_policy.csv",
        "cell_07_final_summary.csv",
        "cell_07_final_validation_gate.csv",
    ],
    "purpose": [
        "Leakage-safe target handoff for Notebook 09",
        "Separate audit containing failure dates and day horizons; never use as predictors",
        "One record for each genuine failed bank and its matching status",
        "All removed same-day, post-failure and incomplete-window observations",
        "One record for each genuine failed bank and its final capture status",
        "Every positive bank-quarter and its validated horizon",
        "Panel exits without a genuine failure; may include mergers or voluntary liquidations",
        "Explicit treatment of failures and non-failure exits",
        "Final Notebook 07 counts and rates",
        "Final fail-fast validation gate",
    ],
})

notebook_07_final_summary = pd.concat([
    notebook_07_final_summary,
    pd.DataFrame({
        "measure": [
            "Analysis version",
            "V2 predictor timing",
            "Target anchor",
            "Fixed final-test reference quarters",
            "Approved target-boundary change",
        ],
        "value": [
            ANALYSIS_VERSION,
            PREDICTOR_TIMING_POLICY,
            "report_date t",
            len(final_test_reference_periods),
            TARGET_BOUNDARY_CHANGE,
        ],
    }),
], ignore_index=True)

notebook_07_output_manifest = pd.concat([
    notebook_07_output_manifest,
    pd.DataFrame({
        "output_file": ["cell_04_v2_calendar_boundary_audit.csv"],
        "purpose": ["Binding V2 outcome-cutoff, target-anchor and 22-quarter test-calendar controls"],
    }),
], ignore_index=True)

cell_07_checks = {
    "The final target contains the locked 589,088 observations":
        (len(notebook_07_primary_target_handoff) == 589088),

    "The final target contains the locked 1,926 positive bank-quarters":
        (int(notebook_07_primary_target_handoff["failed_1yr"].sum()) == 1926),

    "The target captures the locked 486 distinct failed banks":
        (distinct_failed_banks_captured == 486),

    "The V2 final-test calendar remains exactly 22 quarters":
        (len(final_test_reference_periods) == 22),

    "The target remains anchored at report_date t":
        (TARGET_WINDOW_POLICY.endswith("after report_date t")),

    "No target-boundary change is recorded":
        (TARGET_BOUNDARY_CHANGE == "NONE"),

    "The primary handoff contains exactly four columns":
        (
            list(
                notebook_07_primary_target_handoff.columns
            )
            == primary_target_columns
        ),

    "The primary handoff excludes failure dates and horizons":
        (
            {"fail_date", "days_to_failure", "target_reason"}
            .isdisjoint(
                notebook_07_primary_target_handoff.columns
            )
        ),

    "The audit handoff contains all seven locked columns":
        (
            list(
                notebook_07_target_audit_handoff.columns
            )
            == audit_target_columns
        ),

    "No primary CERT-report_date key is duplicated":
        (
            notebook_07_primary_target_handoff.duplicated(
                ["cert", "report_date"]
            ).sum()
            == 0
        ),

    "No primary RSSDID-report_date key is duplicated":
        (
            notebook_07_primary_target_handoff.duplicated(
                ["idrssd", "report_date"]
            ).sum()
            == 0
        ),

    "The final handoff ends in 2025Q2":
        (
            notebook_07_primary_target_handoff[
                "report_date"
            ].max()
            == pd.Timestamp("2025-06-30")
        ),

    "The final handoff contains no missing identifier or target":
        (
            notebook_07_primary_target_handoff[
                primary_target_columns
            ].notna().all().all()
        ),

    "All positive horizons remain within 1-365 days":
        (
            notebook_07_target_audit_handoff.loc[
                notebook_07_target_audit_handoff[
                    "failed_1yr"
                ].eq(1),
                "days_to_failure",
            ].between(1, TARGET_HORIZON_DAYS).all()
        ),

    "Primary and audit handoffs contain identical keys and targets":
        (
            audit_memory_key_hash
            == primary_memory_hash
        ),

    "The primary Parquet output exists":
        (
            PRIMARY_TARGET_PATH.exists()
            and PRIMARY_TARGET_PATH.stat().st_size > 0
        ),

    "The audit Parquet output exists":
        (
            TARGET_AUDIT_PATH.exists()
            and TARGET_AUDIT_PATH.stat().st_size > 0
        ),

    "The primary Parquet output re-read exactly":
        (
            len(primary_recheck)
            == len(notebook_07_primary_target_handoff)
            and primary_file_hash
            == primary_memory_hash
        ),

    "The audit Parquet output re-read with identical keys and target":
        (
            len(audit_recheck)
            == len(notebook_07_target_audit_handoff)
            and audit_file_key_hash
            == audit_memory_key_hash
        ),

    "All previous Notebook 07 gates passed":
        all(
            gate["passed"].all()
            for gate in [
                notebook_07_cell_01_gate,
                notebook_07_cell_02_gate,
                notebook_07_cell_03_gate,
                notebook_07_cell_04_gate,
                notebook_07_cell_05_gate,
                notebook_07_cell_06_gate,
            ]
        ),
}

notebook_07_cell_07_gate = pd.DataFrame({
    "check": list(cell_07_checks.keys()),
    "result": [
        "PASS" if passed else "FAIL"
        for passed in cell_07_checks.values()
    ],
    "passed": list(cell_07_checks.values()),
})

notebook_07_final_summary.to_csv(
    NB07_OUTPUT_DIR / "cell_07_final_summary.csv",
    index=False,
)

notebook_07_output_manifest.to_csv(
    NB07_OUTPUT_DIR / "cell_07_output_manifest.csv",
    index=False,
)

notebook_07_cell_07_gate.to_csv(
    NB07_OUTPUT_DIR / "cell_07_final_validation_gate.csv",
    index=False,
)

print("Notebook 07 Cell 7 validation gate:")
display(notebook_07_cell_07_gate)

print("\nNotebook 07 final handoff summary:")
with pd.option_context("display.max_colwidth", None):
    display(notebook_07_final_summary)

print("\nPrimary leakage-safe target preview:")
display(notebook_07_primary_target_handoff.head(10))

failed_cell_07_checks = [
    check
    for check, passed in cell_07_checks.items()
    if not passed
]

if failed_cell_07_checks:
    raise AssertionError(
        "Notebook 07 Cell 7 blocked. Failed checks: "
        + "; ".join(failed_cell_07_checks)
    )

print("\nCell 7 completed successfully.")
print("Notebook 07 is PASS / LOCKED.")
print("\nPrimary target handoff:")
print(PRIMARY_TARGET_PATH)
print("\nSeparate target audit handoff:")
print(TARGET_AUDIT_PATH)
print("\nNotebook 08 can now prepare the lagged macroeconomic data.")
